# 01 — Exploratory Data Analysis
NASA CMAPSS FD001–FD004 dataset exploration.
Goals: understand sensor distributions, degradation patterns, class imbalance, and inter-variant differences.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import sys; sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import spearmanr

from data.loader import load_fd, load_rul
from labels.rul import add_rul
from labels.binary import add_binary_label

VARIANTS = ['FD001', 'FD002', 'FD003', 'FD004']
SENSOR_COLS = [f'sensor_{i}' for i in range(1, 22)]
print('Imports OK')

## 1. Dataset overview

In [ ]:
summaries = []
for v in VARIANTS:
    df = load_fd(v)
    summaries.append({
        'variant': v,
        'n_engines': df['unit_id'].nunique(),
        'n_rows': len(df),
        'min_cycles': df.groupby('unit_id')['cycle'].max().min(),
        'max_cycles': df.groupby('unit_id')['cycle'].max().max(),
        'median_cycles': df.groupby('unit_id')['cycle'].max().median(),
    })

pd.DataFrame(summaries).set_index('variant')

## 2. Engine lifetime distribution per variant

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=False)
for ax, v in zip(axes, VARIANTS):
    df = load_fd(v)
    lifetimes = df.groupby('unit_id')['cycle'].max()
    ax.hist(lifetimes, bins=20, edgecolor='white', color='steelblue', alpha=0.85)
    ax.set_title(f'{v}\n(n={len(lifetimes)} engines)')
    ax.set_xlabel('Lifetime (cycles)')
    ax.set_ylabel('Count')
plt.suptitle('Engine Lifetime Distribution — CMAPSS', fontsize=13)
plt.tight_layout()
plt.show()

## 3. Sensor degradation trends — FD001 sample engine

In [ ]:
df1 = load_fd('FD001')
# Pick the engine with median lifetime
lifetimes = df1.groupby('unit_id')['cycle'].max()
sample_unit = lifetimes.sub(lifetimes.median()).abs().idxmin()
engine = df1[df1['unit_id'] == sample_unit].sort_values('cycle')

fig, axes = plt.subplots(5, 4, figsize=(18, 14))
for ax, col in zip(axes.flat, SENSOR_COLS[:20]):
    ax.plot(engine['cycle'], engine[col], lw=1, color='steelblue')
    ax.set_title(col, fontsize=9)
    ax.set_xlabel('cycle', fontsize=7)
# Hide unused subplot
axes.flat[-1].set_visible(False)
fig.suptitle(f'FD001 — Engine {sample_unit} sensor traces', fontsize=12)
plt.tight_layout()
plt.show()

## 4. Spearman correlation with RUL — data-driven sensor ranking

In [ ]:
df1 = add_rul(load_fd('FD001'))
corrs = {}
for col in SENSOR_COLS:
    rho, _ = spearmanr(df1[col].dropna(), df1.loc[df1[col].notna(), 'rul'])
    corrs[col] = rho

corr_series = pd.Series(corrs).sort_values()
colors = ['firebrick' if v < 0 else 'steelblue' for v in corr_series]
fig, ax = plt.subplots(figsize=(10, 5))
corr_series.plot.bar(ax=ax, color=colors, edgecolor='white')
ax.axhline(0, color='black', lw=0.8)
ax.set_title('Spearman ρ(sensor, RUL) — FD001 training set')
ax.set_ylabel('Spearman ρ')
ax.set_xlabel('Sensor')
plt.tight_layout()
plt.show()

print('Strongest negative correlation (sensor rises as RUL falls):')
print(corr_series.head(3).to_string())
print('\nSelected trigger sensor (max |ρ|):', corr_series.abs().idxmax())

## 5. Class imbalance at label_within_30

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, v in zip(axes, VARIANTS):
    df = add_binary_label(add_rul(load_fd(v)), x=30)
    counts = df['label_within_x'].value_counts()
    ax.bar(['No fault', 'Fault within 30'], [counts.get(0,0), counts.get(1,0)],
           color=['steelblue', 'firebrick'], edgecolor='white')
    ratio = counts.get(1, 0) / len(df)
    ax.set_title(f'{v}\npositive rate = {ratio:.1%}')
plt.suptitle('Class imbalance — binary label (fail within 30 cycles)', fontsize=12)
plt.tight_layout()
plt.show()

## 6. Operating condition clusters — FD002 (6 conditions)

In [ ]:
from sklearn.preprocessing import KBinsDiscretizer
df2 = load_fd('FD002')
# Discretize op_setting_1 into 6 bins to approximate operating regimes
kbd = KBinsDiscretizer(n_bins=6, encode='ordinal', strategy='kmeans')
df2['op_regime'] = kbd.fit_transform(df2[['op_setting_1', 'op_setting_2']]).astype(int).max(axis=1)

fig, ax = plt.subplots(figsize=(7, 5))
for regime, grp in df2.groupby('op_regime'):
    ax.scatter(grp['op_setting_1'], grp['op_setting_2'], s=2, alpha=0.3, label=f'Regime {regime}')
ax.set_xlabel('op_setting_1')
ax.set_ylabel('op_setting_2')
ax.set_title('FD002 — Operating condition scatter (6 clusters)')
ax.legend(markerscale=4)
plt.tight_layout()
plt.show()